[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/EelcoHoogendoorn/numga/blob/main/examples/mechanics/elastica/elastica.ipynb)

# The Elastica

This example computes the static shapes of Euler and Kirchhoff elastic rods under end loads alongside the rigid-segment dynamics of a hanging, stretched telephone cord. A rod is modelled as a chain of rigid cross-sections. When loads act only at the ends, every cross-section balances the far end load on its own and therefore carries the same spatial force or couple line, bending and twisting in proportion to the moments of that load about its local axes.

Pulling the fixed spatial load into a cross-section frame via `frame << loads` automatically accounts for the shifting moment arm, and applying `rod_compliance(frame << loads)` maps that load to the local twist whose motor exponential advances to the next cross-section. In dynamics, the stiffness dyad sum maps logarithmic twist deviations back to restoring loads through nested commutators of the exponential derivative, while `inertia.solve(forques - momenta.commutator(rates))` advances translational and rotational motion together on one footing ([Gunn, 2011](https://arxiv.org/pdf/1101.4542); [Dorst and De Keninck, 2022](https://bivector.net/PGAdyn.pdf)).

In matrix Cosserat and Newton–Euler rod notation, carrying an end force and couple $(\mathbf{f}_0, \mathbf{m}_0)$ to arc-length $s$ shifts the moment arm by $\mathbf{m}(s) = \mathbf{m}_0 + (\mathbf{r}_0 - \mathbf{r}(s)) \times \mathbf{f}_0$, maps body curvature by $\boldsymbol{\kappa}(s) = B^{-1} R(s)^\top \mathbf{m}(s)$, and integrates the split frame equations $R' = R\,[\boldsymbol{\kappa}]_\times$ and $\mathbf{r}' = R\,\mathbf{e}_3$, while segment dynamics separates into $m\ddot{\mathbf{r}}_i = \Delta\mathbf{f}_i + m\mathbf{g} - c\dot{\mathbf{r}}_i$ and $J\dot{\boldsymbol{\omega}}_i + \boldsymbol{\omega}_i \times (J\boldsymbol{\omega}_i) = R_i^\top\Delta\boldsymbol{\tau}_i - c\,J\boldsymbol{\omega}_i$.

In [ ]:
# In Colab, install numga and its examples from GitHub.
import sys

if "google.colab" in sys.modules:
    %pip install -q "numga[examples] @ https://github.com/EelcoHoogendoorn/numga/archive/HEAD.tar.gz"

In [ ]:
%matplotlib inline
from collections.abc import Iterator

import jax
import numpy as np

from numga import NumpyContext, concatenate, stack
from numga.algebras import PGA3D as ga
from numga.backend.jax import JaxContext
from examples.mechanics.elastica import render
from examples.mechanics.elastica.scenarios import chords, handset_reaches, thrusts
from examples.util.animation import save_figure, show_animation

context = NumpyContext(ga)
mv = context.multivector
Point = ga.gatype.antivector()                             # Antivector-
Twist = ga.gatype.bivector()                               # Bivector+
Forque = ga.gatype.antibivector()                          # Antibivector-
Motor = ga.gatype.rotor()                                  # Rotor+
Compliance = ga.gatype((Twist, Forque))                    # Twist+ <- Forque-
Stiffness = ga.gatype((Forque, Twist))                     # Forque- <- Twist+
Inertia = ga.gatype((Forque, Twist))                       # Forque- <- Twist+

# Cross-section origin, unit axial shift, and local rotation and force axes (across, up, along).
centre = mv.w.dual()                                                                   # [] Point-
shift_along_rod = mv.zw                                                                # [] Twist+
turn_axes = stack([mv.yz, mv.zx, mv.xy])                                               # [3] Twist+
force_axes = stack([centre & mv.x.dual(), centre & mv.y.dual(), centre & mv.z.dual()])  # [3] Forque-

## A rod as a chain of cross-sections

Under loads applied only at the ends, the portion of a rod beyond any cross-section stays in equilibrium under the far end load and the internal load across that slice alone, so every cross-section carries one spatial load line. Pulling that load into the cross-section frame via `frame << loads` resolves its bending and twisting moments about the local axes, and `rod_compliance` scales each axis pairing by the corresponding compliance to form the local twist per unit length. Combining that turn with `shift_along_rod` and exponentiating over each midpoint step multiplies the motors forward along the rod.

In [ ]:
# Static rod length, piece count, bending and twisting compliances, and zero shear/stretch compliances.
length = 1.0
n_pieces = 200
turning_compliance = np.array([1.0, 1.0, 1.4])
shifting_compliance = np.zeros(3)

In [ ]:
def compliance(turning_compliance: np.ndarray, shifting_compliance: np.ndarray) -> Compliance:
    """A dyad sum over the three cross-section axes (`[3]`) maps a carried load to a twist per unit length (`[] Twist+ <- Forque-`)."""
    shift_axes = force_axes.dual()                                                      # [3] Twist+
    turn_terms = turn_axes * (turn_axes & Forque) * turning_compliance                  # [3] Twist+ <- Forque-
    shift_terms = shift_axes * (shift_axes & Forque) * shifting_compliance              # [3] Twist+ <- Forque-
    return (turn_terms + shift_terms).sum(axis=0)                                       # [] Twist+ <- Forque-


def rod_frames(loads: Forque, rod_compliance: Compliance, start_frames: Motor, length: float, n_pieces: int) -> Motor:
    """Midpoint integration across `n_pieces` pieces integrates the carried load into cross-section placements (`[..., n_pieces + 1] Motor+`)."""
    piece_length = length / n_pieces
    frame = start_frames.broadcast_to(np.broadcast_shapes(start_frames.shape, loads.shape))  # [...] Motor+
    frames = [frame]
    for _ in range(n_pieces):
        # Each piece advances along the rod and turns by the load seen halfway across the piece.
        halfway_step = ((shift_along_rod + rod_compliance(frame << loads)) * (piece_length / 4)).exp()  # [...] Motor+
        halfway = frame * halfway_step                                                                  # [...] Motor+
        full_step = ((shift_along_rod + rod_compliance(halfway << loads)) * (piece_length / 2)).exp()   # [...] Motor+
        frame = frame * full_step                                                                       # [...] Motor+
        frames.append(frame)
    return stack(frames, axis=-1)                                                       # [..., n_pieces + 1] Motor+


rod_compliance = compliance(turning_compliance, shifting_compliance)                    # [] Twist+ <- Forque-
rod_compliance

## Curling under a couple

A pure couple about the lateral axis looks unchanged from every cross-section that turns about that same axis. Every piece therefore bends at the same rate, curling the strip into a circular arc that closes into a full ring when the total bend reaches one turn.

In [ ]:
curl_turns = [0.0, 0.25, 0.5, 0.75, 1.0, 2.0]
n_curls = len(curl_turns)

In [ ]:
couples = mv.yz.dual() * (2 * np.pi / (turning_compliance[0] * length) * np.array(curl_turns))  # [n_curls] Forque-
curled_rods = rod_frames(couples, rod_compliance, mv.rotor(), length, n_pieces)                 # [n_curls, n_pieces + 1] Motor+
save_figure(render.draw_curls(curled_rods >> centre), "elastica_curls");

The clamped strip starts straight in the darkest curve, bends through quarter-turn steps into a closed ring, and winds into a double ring at twice that couple.

## Euler's elastica

A rod pinned at both ends can turn freely at each support while an axial thrust along the line between the pins pushes the ends together. Every cross-section carries that fixed line of thrust, so a rod departing the first pin at an oblique angle bends back toward the thrust line and crosses it again at its far end under one critical load, computed alongside the pin separation from elliptic integrals in `thrusts` and `chords`.

In planar elastica coordinates, the tangent angle $\theta(s)$ satisfies $EI\,\theta''(s) + P\sin\theta(s) = 0$ with $(x'(s), y'(s)) = (\cos\theta(s), \sin\theta(s))$, whose first integral yields complete elliptic integrals of $\sin(\alpha/2)$ for the critical thrust and chord length at departure angle $\alpha$.

In [ ]:
n_sweep_frames = 90
sweep_angles = np.linspace(2.0, 176.0, n_sweep_frames)
frame_duration_ms = 40

In [ ]:
pin_thrusts = (centre & mv.z.dual()) * thrusts(np.radians(sweep_angles))                        # [n_sweep_frames] Forque-
pin_starts = (mv.yz * (-np.radians(sweep_angles) / 2)).exp()                                    # [n_sweep_frames] Motor+
sweep_rods = rod_frames(pin_thrusts, rod_compliance, pin_starts, length, n_pieces)              # [n_sweep_frames, n_pieces + 1] Motor+
show_animation(render.animate_sweep(sweep_rods >> centre), "elastica_sweep", frame_duration_ms)

At small departure angles the rod bows into a shallow arch between the grey pins along the line of thrust. As the angle grows, the arch deepens until its ends turn vertical at 90°, the pins meet in a figure eight near 130.7°, and the rod crosses itself in a tightening loop.

## Kirchhoff's helices

An end couple oblique to a wire bends the cross-sections about its axis while twisting them along the wire. When bending compliance is equal in both lateral directions, the tangent precesses at a constant angle around the couple axis so the wire winds into a uniform circular helix, while a flat ribbon carried by the cross-section motors reveals the axial twist.

In [ ]:
tilt_angles = [80.0, 60.0, 40.0, 20.0]
n_tilts = len(tilt_angles)
n_helix_turns = 3
helix_ribbon_width = 0.015

In [ ]:
helix_couple = mv.xy.dual() * (2 * np.pi * n_helix_turns / (turning_compliance[0] * length))    # [] Forque-
helix_starts = (mv.yz * (-np.radians(tilt_angles) / 2)).exp()                                   # [n_tilts] Motor+
coiled_rods = rod_frames(helix_couple, rod_compliance, helix_starts, length, n_pieces)          # [n_tilts, n_pieces + 1] Motor+
helix_offsets = mv.point([[-helix_ribbon_width / 2, 0, 0], [helix_ribbon_width / 2, 0, 0]])     # [2] Point-
helix_edges = coiled_rods[..., None] >> helix_offsets                                           # [n_tilts, n_pieces + 1, 2] Point-
save_figure(render.draw_helices(coiled_rods >> centre, helix_edges), "elastica_helices");

From left to right, the four wires leave the origin at 80°, 60°, 40°, and 20° from the upward couple axis and wind through three full turns into progressively taller, narrower helices.

## Bending into twisting under a tilted couple

Holding the upward couple fixed while sweeping the starting tilt from perpendicular to parallel shifts the load smoothly between pure bending and pure torsion. Each frame integrates the static equilibrium shape at the current tilt while the camera orbits a fixed bounding cylinder around the wire.

In [ ]:
uncoil_turns, n_uncoil_frames, uncoil_ribbon_width = 1.5, 135, 0.035

In [ ]:
uncoil_tilts = 45 * (1 + np.cos(np.linspace(0, 2 * np.pi, n_uncoil_frames, endpoint=False)))    # [n_uncoil_frames]
uncoil_couple = mv.xy.dual() * (2 * np.pi * uncoil_turns / (turning_compliance[0] * length))    # [] Forque-
uncoil_starts = (mv.yz * (-np.radians(uncoil_tilts) / 2)).exp()                                 # [n_uncoil_frames] Motor+
uncoiled_rods = rod_frames(uncoil_couple, rod_compliance, uncoil_starts, length, n_pieces)      # [n_uncoil_frames, n_pieces + 1] Motor+
uncoil_offsets = mv.point([[-uncoil_ribbon_width / 2, 0, 0], [uncoil_ribbon_width / 2, 0, 0]])  # [2] Point-
uncoiled_edges = uncoiled_rods[..., None] >> uncoil_offsets                                     # [n_uncoil_frames, n_pieces + 1, 2] Point-
show_animation(render.animate_uncoiling(uncoiled_rods >> centre, uncoiled_edges), "elastica_uncoiling", frame_duration_ms)

Perpendicular to the couple, the wire closes into a flat ring with an untwisted ribbon. As the tilt decreases toward the couple axis, the ring opens into a stretching helix and straightens into a pure axial rod whose ribbon wrings through extra turns because twisting compliance exceeds bending compliance.

## Pulling a telephone cord

A telephone cord already carries bend and twist in its unstressed coil, modelled dynamically as rigid segments with inertia linked by elastic joints around that helical rest step. Each joint maps its logarithmic twist deviation through `stiffness` and the right-trivialized exponential derivative so the joint loads remain conservative, while `cord_step` advances interior segments under joint loads, gravity, damping, and the gyroscopic commutator `momenta.commutator(rates)`. Stepping and rendering the 70-frame pull-and-release animation takes around ten seconds.

In [ ]:
cord_length, n_cord_coils, n_segments_per_coil, cord_pitch = 1.0, 5, 6, 10.0
cord_turning_rigidity = np.array([1e-3, 1e-3, 7e-4])
cord_shifting_rigidity = np.array([20.0, 20.0, 20.0])
cord_mass, cord_spread, gravity, cord_damping = 0.05, 2.0, 9.81, 1.0
hang_time, pull_time, handset_reach, hold_time, release_time = 0.3, 0.9, 0.6, 0.3, 0.4
time_step, n_steps_per_frame, n_cord_frames = 1e-3, 40, 70
n_subdivisions, cord_ribbon_width = 4, 0.025

In [ ]:
def stiffness(turning_rigidity: np.ndarray, shifting_rigidity: np.ndarray) -> Stiffness:
    """A dyad sum over the cross-section couple and force axes (`[3]`) maps a twist per unit length to a restoring load (`[] Forque- <- Twist+`)."""
    couple_axes = turn_axes.dual()                                                      # [3] Forque-
    turn_terms = couple_axes * (Twist & couple_axes) * turning_rigidity                 # [3] Forque- <- Twist+
    shift_terms = force_axes * (Twist & force_axes) * shifting_rigidity                 # [3] Forque- <- Twist+
    return (turn_terms + shift_terms).sum(axis=0)                                       # [] Forque- <- Twist+


def segment_inertia(mass: float, length: float, radius: float) -> Inertia:
    """Six point masses across a segment's ends and mid-ring (`[6]`) define its rigid-body inertia (`[] Forque- <- Twist+`)."""
    points = mv.point([
        [radius, 0, 0], [-radius, 0, 0], [0, radius, 0], [0, -radius, 0],
        [0, 0, length / 2], [0, 0, -length / 2],
    ])                                                                                  # [6] Point-
    return ((mass / 6) * points).regressive(points.commutator(Twist)).sum(axis=0)       # [] Forque- <- Twist+

In [ ]:
def joint_loads(frames: Motor, rest_step: Motor, joint_stiffness: Stiffness, piece_length: float) -> Forque:
    """Each joint maps its relative twist deviation from `rest_step` to the world-frame load on the following segment (`[...] Forque-`)."""
    deviation = (rest_step.reverse() * frames[..., :-1].reverse() * frames[..., 1:]).log() * 2  # [...] Twist+
    local_load = joint_stiffness(deviation / piece_length)                                      # [...] Forque-
    # The right-trivialized exponential derivative keeps the joint load equal to the energy gradient.
    restoring_load = (
        local_load
        - deviation.commutator(local_load) / 2
        + deviation.commutator(deviation.commutator(local_load)) / 12
    )                                                                                           # [...] Forque-
    return frames[..., 1:] >> restoring_load                                                    # [...] Forque-


def cord_step(
    frames: Motor,
    rates: Twist,
    rest_step: Motor,
    joint_stiffness: Stiffness,
    piece_length: float,
    inertia: Inertia,
    weight: Point,
    damping: float,
    time_step: float,
) -> tuple[Motor, Twist]:
    """Euler–Poincaré integration advances the interior segments and body rates by `time_step` between held end frames."""
    loads = joint_loads(frames, rest_step, joint_stiffness, piece_length)               # [...] Forque-
    segments = frames[1:-1]                                                             # [...] Motor+
    momenta = inertia(rates)                                                            # [...] Forque-
    # Gravity acts along the vertical line through each segment's centre, while damping opposes its momentum.
    forques = (segments << (loads[:-1] - loads[1:])) + (centre & (segments << weight)) - damping * momenta  # [...] Forque-
    next_rates = rates + inertia.solve(forques - momenta.commutator(rates)) * time_step                     # [...] Twist+
    step_motors = (next_rates * (-time_step / 2)).exp()                                                     # [...] Motor+
    next_segments = segments * step_motors                                                                  # [...] Motor+
    return concatenate([frames[:1], next_segments, frames[-1:]]), next_rates

In [ ]:
def cord_trajectory(
    rest_frames: Motor,
    rest_step: Motor,
    coil_shift: Twist,
    joint_stiffness: Stiffness,
    piece_length: float,
    inertia: Inertia,
    weight: Point,
    damping: float,
    time_step: float,
    reaches: np.ndarray,
) -> Iterator[Motor]:
    """Yields the segment frames (`[...] Motor+`) at each frame of the handset displacement schedule `reaches`."""
    n_segments = len(rest_frames) - 2
    jax_context = JaxContext(ga)

    def scan_step(state: tuple[Motor, Twist], reach: jax.Array) -> tuple[tuple[Motor, Twist], None]:
        frames, rates = state
        # Shifting the handset along the coil axis updates the far boundary before stepping the interior segments.
        handset_shift = (coil_shift * (reach / 2)).exp()                                # [] Motor+
        handset_frame = handset_shift * rest_frames[-1]                                 # [] Motor+
        next_frames = concatenate([frames[:-1], handset_frame[None]])                   # [...] Motor+
        next_state = cord_step(
            next_frames, rates, rest_step, joint_stiffness, piece_length,
            inertia, weight, damping, time_step,
        )
        return next_state, None

    advance_frame = jax.jit(lambda state, frame_reaches: jax.lax.scan(scan_step, state, frame_reaches)[0])
    initial_rates = (0 * shift_along_rod).broadcast_to((n_segments,)).retyped(Twist)    # [n_segments] Twist+
    state = (
        rest_frames.retyped(Motor).materialize(jax_context),
        initial_rates.materialize(jax_context),
    )
    for frame_reaches in reaches:
        yield state[0].materialize(context)
        state = advance_frame(state, frame_reaches)

In [ ]:
def dense_frames(frames: Motor, n_subdivisions: int) -> Motor:
    """Logarithmic relative-twist interpolation inserts `n_subdivisions` equal steps across each joint (`[...] Motor+`)."""
    relative_twists = (frames[..., :-1].reverse() * frames[..., 1:]).log()              # [...] Twist+
    shares = np.arange(n_subdivisions) / n_subdivisions                                 # [n_subdivisions]
    step_motors = (relative_twists[..., None] * shares).exp()                           # [..., n_subdivisions] Motor+
    parts = frames[..., :-1, None] * step_motors                                        # [..., n_subdivisions] Motor+
    return concatenate([parts.reshape(parts.shape[:-2] + (-1,)), frames[..., -1:]], axis=-1)  # [...] Motor+


n_segments = n_cord_coils * n_segments_per_coil
n_drawn = (n_segments + 1) * n_subdivisions + 1
piece_length = cord_length / n_segments
# Tilting the lateral bend toward the rod axis creates the cord's helical rest twist and axial pull shift.
pitch_tilt = (mv.xz * (-np.radians(cord_pitch) / 2)).exp()                              # [] Motor+
coiling_twist = shift_along_rod + (pitch_tilt >> mv.yz) * (2 * np.pi * n_cord_coils / cord_length)  # [] Twist+
rest_frames = (coiling_twist * (np.arange(-1, n_segments + 1) * piece_length / 2)).exp()            # [n_segments + 2] Motor+
rest_step = (coiling_twist * (piece_length / 2)).exp()                                              # [] Motor+
coil_shift = pitch_tilt >> mv.xw                                                                    # [] Twist+
joint_stiffness = stiffness(cord_turning_rigidity, cord_shifting_rigidity)                          # [] Forque- <- Twist+
segment_mass = cord_mass * piece_length
inertia = segment_inertia(segment_mass, piece_length, cord_spread * piece_length)                   # [] Forque- <- Twist+
weight = mv.y.dual() * (-gravity * segment_mass)                                                    # [] Point-
reaches = handset_reaches(
    n_cord_frames, n_steps_per_frame, time_step, hang_time, pull_time, handset_reach, hold_time, release_time,
)                                                                                                   # [n_cord_frames, n_steps_per_frame]
moved_cord = stack(list(cord_trajectory(
    rest_frames, rest_step, coil_shift, joint_stiffness, piece_length,
    inertia, weight, cord_damping, time_step, reaches,
)))                                                                                                 # [n_cord_frames, n_segments + 2] Motor+
drawn_cord = dense_frames(moved_cord, n_subdivisions)                                               # [n_cord_frames, n_drawn] Motor+
cord_offsets = mv.point([[-cord_ribbon_width / 2, 0, 0], [cord_ribbon_width / 2, 0, 0]])            # [2] Point-
cord_edges = drawn_cord[..., None] >> cord_offsets                                                  # [n_cord_frames, n_drawn, 2] Point-
show_animation(render.animate_cord(drawn_cord >> centre, cord_edges), "elastica_cord", frame_duration_ms)

The cord first sags under its own weight, stretches into an open helix as the handset pulls away along the coil axis, settles while held, and tumbles back into rebounding coils upon release until damping calms the motion.

In [ ]:
# checks
# Whole-turn couples return the curled strip's free end to the clamp, while zero couple leaves it straight.
curl_ends = curled_rods[:, -1] >> centre                                                        # [n_curls] Point-
assert (curl_ends[0] & (centre + mv.z.dual() * length)).norm() < 1e-10
assert ((curl_ends[[4, 5]] & centre).norm() < 1e-10).all()
# Every Euler elastica returns to the line of thrust at the elliptic-integral chord distance from the first pin.
expected_pins = centre + mv.z.dual() * chords(np.radians(sweep_angles))                         # [n_sweep_frames] Point-
assert (((sweep_rods[:, -1] >> centre) & expected_pins).norm() < 1e-3).all()
# Three full helical turns bring each wire's far end directly above the start at height L cos(tilt).
expected_tips = centre + mv.z.dual() * (length * np.cos(np.radians(tilt_angles)))               # [n_tilts] Point-
assert (((coiled_rods[:, -1] >> centre) & expected_tips).norm() < 3e-3).all()
# High shear and stretch rigidities keep relative joint shifts small compared with segment length.
deviations = (rest_step.reverse() * moved_cord[:, :-1].reverse() * moved_cord[:, 1:]).log() * 2  # [n_cord_frames, n_segments + 1] Twist+
assert (((deviations[..., None] & force_axes) / piece_length).abs() < 0.1).all()